# SA-SSM Plugin Training for Object Detection

**Paper**: SA-SSM: Structurally Noise-Robust State Space Model Plugin for Always-On Object Detection

**Architecture**: YOLOv8n + SA-SSM Plugin (0.15M~0.92M overhead)

**4 Structural Defenses**: LTI Stability | DCT Spectral Analysis | Gate Floor | Heterogeneous Expert Routing

---

## Training Strategy
- **Stage 1** (15 epochs): Freeze backbone+neck, train only SA-SSM plugin
- **Stage 2** (85 epochs): Unfreeze all, fine-tune with corruption augmentation

## Profiles
| Profile | Overhead | Target |
|---------|----------|--------|
| Edge | 0.39M (11%) | Jetson Orin |
| Tiny | 0.15M (4%) | Smart Glasses |
| Ultra-lite | 0.22M (6%) | Jetson Nano |

## 0. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    # Auto-detect batch size
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    BATCH_SIZE = 32 if vram_gb > 30 else 16 if vram_gb > 12 else 8
    print(f"Recommended batch_size: {BATCH_SIZE}")
else:
    raise RuntimeError("GPU not available! Go to Runtime > Change runtime type > GPU")

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics>=8.1.0 pycocotools>=2.0.7 einops>=0.7.0 tensorboard
print("Dependencies installed.")

# Verify
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

## 2. Clone NAS-YOLO Repository

In [ ]:
import os

REPO_URL = "https://github.com/DrJinHoChoi/NAS-YOLO.git"
REPO_DIR = "/content/NAS-YOLO"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Add to Python path
import sys
sys.path.insert(0, REPO_DIR)

## 3. Mount Google Drive (for checkpoint saving)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/SA-SSM_runs"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Results will be saved to: {DRIVE_OUTPUT}")

---
## 4. Smoke Test (COCO128, 2 epochs, ~5 min)

Quick sanity check: download COCO128 (128 images) and run 2 epochs.

In [ ]:
# Download COCO128 via ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

# This downloads coco128 to ~/datasets/coco128
data_info = check_det_dataset("coco128.yaml")
print(f"COCO128 downloaded to: {data_info}")

In [ ]:
# === SMOKE TEST: SA-SSM Plugin ===
import torch
import time
from nas_yolo.integrations.ultralytics_wrapper import UltralyticsWithNASPlugin

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build model: YOLOv8n + SA-SSM Edge plugin
print("Building YOLOv8n + SA-SSM Edge plugin...")
model = UltralyticsWithNASPlugin(
    weights_path="yolov8n.pt",
    plugin_cfg={
        "profile": "edge",
        "mode": "hybrid",
        "use_noise_gate": True,
        "noise_dim": 32,
    },
    freeze_backbone=True,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
plugin_params = sum(p.numel() for p in model.nas_plugin.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Plugin params: {plugin_params:,} ({plugin_params/total_params*100:.1f}%)")
print(f"Trainable (Stage 1): {trainable:,}")

# Forward pass test
x = torch.randn(2, 3, 640, 640, device=device)
with torch.no_grad():
    t0 = time.time()
    out = model(x)
    t1 = time.time()
print(f"\nForward pass: {(t1-t0)*1000:.1f} ms")
print(f"Output type: {type(out)}")
if isinstance(out, dict):
    for k, v in out.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.shape}")
        else:
            print(f"  {k}: {type(v)}")

print("\n=== Smoke test PASSED ===")

In [ ]:
# === Structural Defense Verification ===
print("=" * 60)
print("  4-Defense Structural Verification")
print("=" * 60)

from nas_yolo.models.structural_theory import StructuralNoiseDefenseAnalyzer

analyzer = StructuralNoiseDefenseAnalyzer(model.nas_plugin)
report = analyzer.full_report()

for defense_name, defense_data in report["defenses"].items():
    status = "ACTIVE" if (
        defense_data.get("all_stable", False) or
        defense_data.get("estimator_found", False) or
        defense_data.get("all_have_floor", False) or
        defense_data.get("has_dual_branch", False)
    ) else "INACTIVE"
    print(f"  D{list(report['defenses'].keys()).index(defense_name)+1}: {defense_data['defense']:<30} [{status}]")

print(f"\n  Summary: {report['summary']['defenses_active']}/{report['summary']['defenses_total']} defenses active")
print(f"  Plugin info: {model.nas_plugin.get_plugin_info()['total_params']:,} params")

---
## 5. COCO 2017 Download (Full Dataset, ~20GB)

**Skip this cell** if you only want to run the smoke test.

In [ ]:
%%time
# Download COCO 2017 train/val
import os

COCO_DIR = "/content/datasets/coco"
os.makedirs(COCO_DIR, exist_ok=True)

# Download images
if not os.path.exists(f"{COCO_DIR}/train2017"):
    print("Downloading COCO train2017 (~18GB)...")
    !wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/train2017.zip
    !unzip -q /content/train2017.zip -d {COCO_DIR}
    !rm /content/train2017.zip
    print(f"train2017: {len(os.listdir(f'{COCO_DIR}/train2017'))} images")
else:
    print(f"train2017 already exists: {len(os.listdir(f'{COCO_DIR}/train2017'))} images")

if not os.path.exists(f"{COCO_DIR}/val2017"):
    print("Downloading COCO val2017 (~1GB)...")
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
    !unzip -q /content/val2017.zip -d {COCO_DIR}
    !rm /content/val2017.zip
    print(f"val2017: {len(os.listdir(f'{COCO_DIR}/val2017'))} images")

# Download annotations
if not os.path.exists(f"{COCO_DIR}/annotations"):
    print("Downloading COCO annotations...")
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/ann.zip
    !unzip -q /content/ann.zip -d {COCO_DIR}
    !rm /content/ann.zip

print(f"\nCOCO 2017 ready at: {COCO_DIR}")
print(f"  train: {len(os.listdir(f'{COCO_DIR}/train2017'))} images")
print(f"  val: {len(os.listdir(f'{COCO_DIR}/val2017'))} images")

---
## 6. Training: SA-SSM Edge Profile (100 epochs)

**Stage 1** (epochs 1-15): Backbone frozen, only plugin trains

**Stage 2** (epochs 16-100): All parameters unfrozen, fine-tune with lower LR

In [ ]:
# === Training Configuration ===
import yaml

PROFILE = "edge"  # Change to "tiny" or "ultra-lite" as needed
CONFIG_PATH = f"colab/configs/colab_{PROFILE}.yaml"

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

# Auto-adjust batch size based on GPU
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
if vram_gb > 30:  # A100
    config["training"]["batch_size"] = 32
    config["training"]["grad_accum_steps"] = 1
    config["plugin"]["gradient_checkpointing"] = False
elif vram_gb > 12:  # T4 / V100
    config["training"]["batch_size"] = 16
    config["training"]["grad_accum_steps"] = 2
    config["plugin"]["gradient_checkpointing"] = True
else:  # Low VRAM
    config["training"]["batch_size"] = 8
    config["training"]["grad_accum_steps"] = 4
    config["plugin"]["gradient_checkpointing"] = True

# Update data paths for Colab
config["data"]["train"] = "/content/datasets/coco/train2017"
config["data"]["val"] = "/content/datasets/coco/val2017"
config["output"]["dir"] = f"/content/NAS-YOLO/runs/yolov8n_{PROFILE}"

print("=" * 50)
print(f"  Training Config: {PROFILE} profile")
print("=" * 50)
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Grad accum: {config['training']['grad_accum_steps']}")
print(f"  Effective batch: {config['training']['batch_size'] * config['training']['grad_accum_steps']}")
print(f"  Stage 1: {config['training']['stage1_epochs']} epochs (backbone frozen)")
print(f"  Stage 2: {config['training']['stage2_epochs']} epochs (all unfrozen)")
print(f"  AMP: {config['training']['amp']}")
print(f"  Grad checkpoint: {config['plugin']['gradient_checkpointing']}")
print(f"  VRAM: {vram_gb:.1f} GB")

In [ ]:
# === BUILD MODEL ===
from nas_yolo.integrations.ultralytics_wrapper import UltralyticsWithNASPlugin
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import time
import os

device = "cuda"

# Build model
model = UltralyticsWithNASPlugin(
    weights_path=config["model"]["pretrained_weights"],
    plugin_cfg=config["plugin"],
    freeze_backbone=True,  # Stage 1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
plugin_params = sum(p.numel() for p in model.nas_plugin.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model built: {total_params:,} total params")
print(f"  Plugin: {plugin_params:,} ({plugin_params/total_params*100:.1f}%)")
print(f"  Stage 1 trainable: {trainable:,}")

In [ ]:
# === BUILD DATASET & DATALOADER ===
from nas_yolo.data.dataset import COCODetectionDataset, build_dataloader
from nas_yolo.data.transforms import DetectionTransform
from nas_yolo.data.corruption import CorruptionTransform

img_size = config["data"]["img_size"]
train_transform = DetectionTransform(img_size=(img_size, img_size), augment=True)

corruption = None
if config["augmentation"].get("corruption_aug", False):
    corruption = CorruptionTransform(p=config["augmentation"]["corruption_p"])

train_dataset = COCODetectionDataset(
    root_dir=config["data"]["train"],
    ann_file="/content/datasets/coco/annotations/instances_train2017.json",
    transform=train_transform,
    corruption=corruption,
    img_size=(img_size, img_size),
)

val_dataset = COCODetectionDataset(
    root_dir=config["data"]["val"],
    ann_file="/content/datasets/coco/annotations/instances_val2017.json",
    transform=DetectionTransform(img_size=(img_size, img_size), augment=False),
    img_size=(img_size, img_size),
)

train_loader = build_dataloader(
    train_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=True,
    num_workers=4,
)

val_loader = build_dataloader(
    val_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    num_workers=4,
)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Val: {len(val_dataset)} images, {len(val_loader)} batches")

In [ ]:
# === TRAINING LOOP ===
from nas_yolo.scripts.train_plugin import (
    LinearWarmupCosineAnnealing,
    train_one_epoch,
)

# Output dir
output_dir = config["output"]["dir"]
os.makedirs(output_dir, exist_ok=True)

# Training settings
stage1_epochs = config["training"]["stage1_epochs"]
stage2_epochs = config["training"]["stage2_epochs"]
total_epochs = stage1_epochs + stage2_epochs
base_lr = config["training"]["base_lr"]
amp_enabled = config["training"]["amp"]
grad_accum = config["training"]["grad_accum_steps"]
max_grad_norm = config["training"]["clip_grad_norm"]

# Optimizer & Scheduler
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=base_lr,
    weight_decay=config["training"]["weight_decay"],
)
scheduler = LinearWarmupCosineAnnealing(
    optimizer,
    warmup_epochs=config["training"]["warmup_epochs"],
    total_epochs=total_epochs,
)
scaler = GradScaler(enabled=amp_enabled)

# Training
best_loss = float("inf")
history = {"epoch": [], "loss": [], "stage": []}

print("\n" + "=" * 60)
print(f"  SA-SSM Training: {PROFILE} profile")
print(f"  Stage 1: {stage1_epochs} epochs (plugin only)")
print(f"  Stage 2: {stage2_epochs} epochs (full fine-tune)")
print("=" * 60)

for epoch in range(total_epochs):
    # Stage transition
    if epoch == stage1_epochs:
        print("\n>>> Stage 2: Unfreezing backbone+neck <<<")
        if hasattr(model, 'unfreeze_all'):
            model.unfreeze_all()
        else:
            for p in model.parameters():
                p.requires_grad = True
        # Rebuild optimizer with all params
        optimizer = optim.AdamW(
            model.parameters(),
            lr=base_lr * 0.1,  # Lower LR for Stage 2
            weight_decay=config["training"]["weight_decay"],
        )
        scheduler = LinearWarmupCosineAnnealing(
            optimizer,
            warmup_epochs=1,
            total_epochs=stage2_epochs,
        )
        scaler = GradScaler(enabled=amp_enabled)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")

    stage = 1 if epoch < stage1_epochs else 2
    t0 = time.time()

    avg_loss = train_one_epoch(
        model, train_loader, optimizer, scaler, device,
        epoch, amp_enabled,
        grad_accum_steps=grad_accum,
        max_grad_norm=max_grad_norm,
    )

    scheduler.step()
    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch + 1)
    history["loss"].append(avg_loss)
    history["stage"].append(stage)

    print(f"Epoch {epoch+1:3d}/{total_epochs} | Stage {stage} | "
          f"Loss: {avg_loss:.4f} | LR: {lr:.6f} | Time: {elapsed:.0f}s")

    # Save checkpoint
    save_every = config["output"].get("save_every", 10)
    if (epoch + 1) % save_every == 0 or (epoch + 1) == total_epochs:
        ckpt_path = os.path.join(output_dir, f"checkpoint_ep{epoch+1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
            "config": config,
        }, ckpt_path)
        print(f"  Saved: {ckpt_path}")

        # Also save to Google Drive
        drive_ckpt = os.path.join(DRIVE_OUTPUT, f"{PROFILE}_ep{epoch+1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "loss": avg_loss,
            "config": config,
        }, drive_ckpt)
        print(f"  Drive: {drive_ckpt}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(output_dir, "best.pt")
        torch.save(model.state_dict(), best_path)

print(f"\nTraining complete! Best loss: {best_loss:.4f}")
print(f"Best model: {best_path}")

---
## 7. Evaluation: Clean + Corruption Robustness

In [ ]:
# === EVALUATION ===
from nas_yolo.engine.evaluator import Evaluator
import json

# Load best model
best_path = os.path.join(output_dir, "best.pt")
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

# Evaluate
evaluator = Evaluator(model, val_loader, device=device)

print("\n" + "=" * 60)
print("  Clean Evaluation")
print("=" * 60)
clean_results = evaluator.evaluate()
print(f"  mAP@0.5: {clean_results.get('mAP50', 'N/A')}")
print(f"  mAP@0.5:0.95: {clean_results.get('mAP50_95', 'N/A')}")

# Corruption evaluation
print("\n" + "=" * 60)
print("  Corruption Robustness Evaluation")
print("=" * 60)
corruption_types = ["gaussian_noise", "motion_blur", "fog", "contrast", "jpeg_compression"]
severities = [1, 3, 5]

corruption_results = {}
for ctype in corruption_types:
    corruption_results[ctype] = {}
    for sev in severities:
        try:
            result = evaluator.evaluate_corruption(ctype, severity=sev)
            corruption_results[ctype][sev] = result.get('mAP50', 0)
            print(f"  {ctype} sev={sev}: mAP@0.5 = {result.get('mAP50', 'N/A')}")
        except Exception as e:
            print(f"  {ctype} sev={sev}: Error - {e}")
            corruption_results[ctype][sev] = None

# Save results
results = {
    "profile": PROFILE,
    "clean": clean_results,
    "corruption": corruption_results,
    "config": config,
}
results_path = os.path.join(output_dir, "eval_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nResults saved: {results_path}")

---
## 8. Profile Comparison (Optional)

Compare Edge vs Tiny vs Ultra-lite overhead and inference speed.

In [ ]:
# === Profile Comparison ===
import torch
import time
from nas_yolo.models.nas_plugin import NASPlugin

profiles = ["standard", "lite", "edge", "ultra-lite", "tiny"]
channels = [64, 128, 256]
x_dummy = [
    torch.randn(1, 64, 80, 80, device=device),
    torch.randn(1, 128, 40, 40, device=device),
    torch.randn(1, 256, 20, 20, device=device),
]
img_dummy = torch.randn(1, 3, 640, 640, device=device)

print(f"{'Profile':<15} {'Params':>10} {'Overhead':>10} {'Latency':>10}")
print("-" * 50)

for pname in profiles:
    plugin = NASPlugin(channels, profile=pname, mode="hybrid").to(device).eval()
    params = sum(p.numel() for p in plugin.parameters())

    # Warmup
    with torch.no_grad():
        for _ in range(3):
            plugin(x_dummy, images=img_dummy)

    # Benchmark
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(20):
            plugin(x_dummy, images=img_dummy)
    torch.cuda.synchronize()
    lat = (time.time() - t0) / 20 * 1000

    print(f"{pname:<15} {params:>10,} {params/3.2e6*100:>9.1f}% {lat:>9.1f}ms")
    del plugin
    torch.cuda.empty_cache()

---
## 9. ONNX Export

In [ ]:
# === ONNX Export ===
import torch.onnx

model.eval()
model.nas_plugin.reset_state()

dummy_input = torch.randn(1, 3, 640, 640, device=device)
onnx_path = os.path.join(output_dir, f"yolov8n_sassm_{PROFILE}.onnx")

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        opset_version=17,
        input_names=["images"],
        output_names=["detections"],
        dynamic_axes={"images": {0: "batch"}, "detections": {0: "batch"}},
    )
    onnx_size = os.path.getsize(onnx_path) / 1e6
    print(f"ONNX exported: {onnx_path} ({onnx_size:.1f} MB)")
except Exception as e:
    print(f"ONNX export failed: {e}")
    print("(This is expected if the model uses dynamic SSM state. Use torch.jit.trace instead.)")

---
## 10. Save Results to Google Drive

In [ ]:
# Copy all results to Drive
import shutil

drive_profile_dir = os.path.join(DRIVE_OUTPUT, PROFILE)
os.makedirs(drive_profile_dir, exist_ok=True)

# Copy key files
for fname in os.listdir(output_dir):
    src = os.path.join(output_dir, fname)
    dst = os.path.join(drive_profile_dir, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"  Copied: {fname}")

# Save training history
import json
with open(os.path.join(drive_profile_dir, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

print(f"\nAll results saved to: {drive_profile_dir}")
print("You can download from Google Drive or access in future sessions.")

---
## 11. Training Loss Plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

epochs = history["epoch"]
losses = history["loss"]
stages = history["stage"]

# Color by stage
s1_ep = [e for e, s in zip(epochs, stages) if s == 1]
s1_loss = [l for l, s in zip(losses, stages) if s == 1]
s2_ep = [e for e, s in zip(epochs, stages) if s == 2]
s2_loss = [l for l, s in zip(losses, stages) if s == 2]

ax.plot(s1_ep, s1_loss, 'b-o', markersize=3, label="Stage 1 (plugin only)")
ax.plot(s2_ep, s2_loss, 'r-o', markersize=3, label="Stage 2 (full fine-tune)")
ax.axvline(x=len(s1_ep), color='gray', linestyle='--', alpha=0.5, label="Stage transition")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"SA-SSM {PROFILE} Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(output_dir, "training_loss.png")
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")